<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/notebooks/02_validation_quantification.ipynb)

*Click the badge to open this notebook in Google Colab. For best performance, switch to a GPU runtime: Runtime → Change runtime type → T4 GPU.*

# Notebook 02 — Validation and Quantification (Lab 2)

**Lab time.** 75 minutes.
**Prerequisites.** Notebook 01 completed; segmentation outputs saved.

**Learning goals.**

1. Compute pixel-level (IoU, Dice) and instance-level (precision, recall) metrics.
2. Compare predicted segmentations to ground truth.
3. Demonstrate the **metrics versus biology** gap: a model can have high IoU and still produce wrong biological measurements.
4. Choose validation metrics that match the biological question.

The lab's central learning moment is in Step 5 — pay attention there.

> **A note on the form widgets.** Several cells below use `#@param` comments. In **Google Colab** these render as interactive form widgets (sliders, dropdowns) at the top of the cell. In **other environments** (JupyterLab, VS Code, the JB rendered HTML) they appear as plain Python comments — edit the values directly and re-run the cell.

## Setup and load Lab 1 outputs

In [ ]:
%pip install --quiet numpy scikit-image matplotlib pandas seaborn scipy
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Load Lab 1 outputs. If they don't exist, fall back to synthetic data so
# the lab can still be completed.
have_lab1 = all(os.path.exists(p) for p in ["masks_easy.npy", "masks_hard.npy"])
if have_lab1:
    masks_easy = np.load("masks_easy.npy")
    masks_hard = np.load("masks_hard.npy")
    print("Loaded Lab 1 outputs.")
else:
    print("Lab 1 outputs not found. Generating synthetic data for the lab.")
    # Synthetic fallback: a 'predicted' segmentation we can score
    rng = np.random.default_rng(42)
    masks_easy = np.zeros((200, 200), dtype=np.int32)
    for i in range(8):
        cy, cx = rng.integers(20, 180, size=2)
        r = rng.integers(10, 18)
        Y, X = np.ogrid[:200, :200]
        masks_easy[(Y - cy)**2 + (X - cx)**2 <= r**2] = i + 1
    masks_hard = masks_easy.copy()
    masks_hard[masks_hard == 3] = 2  # merge two cells
    masks_hard[masks_hard == 6] = 0  # delete a cell

print(f"masks_easy: {masks_easy.shape}, {int(masks_easy.max())} objects")
print(f"masks_hard: {masks_hard.shape}, {int(masks_hard.max())} objects")

### Visualization helpers

We define three small helpers used throughout the lab. Run this cell once; every subsequent visualization calls them.

In [ ]:
from matplotlib.colors import ListedColormap

# Categorical colormap for instance labels (background black, then qualitative)
_label_cmap = ListedColormap(['black'] + plt.get_cmap('tab20')(np.linspace(0, 1, 20)).tolist())

def show_mask(ax, mask, title="", cmap=_label_cmap):
    """Render an integer-label mask with one color per instance."""
    n_lbl = max(int(mask.max()), 1)
    ax.imshow(mask, cmap=cmap, vmin=0, vmax=20)
    ax.set_title(f"{title}  ({n_lbl} obj)" if title else f"{n_lbl} obj")
    ax.axis('off')

def pixel_agreement_rgb(pred, gt):
    """Color-code per-pixel agreement: green=TP, red=FP, blue=FN."""
    pred_bin = pred > 0
    gt_bin = gt > 0
    rgb = np.zeros((*pred.shape, 3), dtype=float)
    rgb[..., 0] = (pred_bin & ~gt_bin).astype(float)   # red   = FP
    rgb[..., 1] = (pred_bin & gt_bin).astype(float)    # green = TP
    rgb[..., 2] = (~pred_bin & gt_bin).astype(float)   # blue  = FN
    return rgb

def instance_match_status(pred, gt, iou_threshold=0.5):
    """Return per-cell match status for both pred and gt.

    Returns (pred_status, gt_status) as integer images:
      pred_status: 0=bg, 1=TP, 2=FP
      gt_status:   0=bg, 1=TP, 3=FN  (3 just to keep colors distinct)
    """
    pred_ids = [i for i in np.unique(pred) if i != 0]
    gt_ids   = [i for i in np.unique(gt) if i != 0]
    matched_pred, matched_gt = set(), set()
    for p in pred_ids:
        pb = (pred == p); best_iou, best_g = 0.0, None
        for g in gt_ids:
            if g in matched_gt: continue
            gb = (gt == g)
            inter = (pb & gb).sum(); union = (pb | gb).sum()
            if union == 0: continue
            iou_val = inter / union
            if iou_val > best_iou:
                best_iou, best_g = iou_val, g
        if best_iou >= iou_threshold:
            matched_pred.add(p); matched_gt.add(best_g)
    pred_status = np.zeros_like(pred, dtype=np.uint8)
    gt_status   = np.zeros_like(gt,   dtype=np.uint8)
    for p in pred_ids:
        pred_status[pred == p] = 1 if p in matched_pred else 2   # TP / FP
    for g in gt_ids:
        gt_status[gt == g] = 1 if g in matched_gt else 3         # TP / FN
    return pred_status, gt_status

print("Helpers defined: show_mask, pixel_agreement_rgb, instance_match_status.")

## Generate ground truth

Real ground truth is hard. For this lab we construct a synthetic ground truth that we *know* is correct, so we can measure exactly how the predictions deviate. In practice the ground truth would come from expert annotation.

In [ ]:
# Use the easy mask as the 'truth' for both images, simulating a case where
# the model overfits the easy distribution and degrades on the harder one.
gt_easy = masks_easy.copy()

# For the hard ground truth, we'll perturb less aggressively — pretending the
# real biology has more cells than the model found.
gt_hard = masks_easy.copy()  # in this synthetic setup, ground truth is the same as easy
print(f"gt_easy: {int(gt_easy.max())} objects")
print(f"gt_hard: {int(gt_hard.max())} objects")

**Visualize what we're comparing.** Before any metric, see the four masks side-by-side. Top row = ground truth; bottom row = prediction. Differences should be visible by eye on the hard column.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 9))
show_mask(axes[0, 0], gt_easy,    "Easy — ground truth")
show_mask(axes[0, 1], gt_hard,    "Hard — ground truth")
show_mask(axes[1, 0], masks_easy, "Easy — predicted")
show_mask(axes[1, 1], masks_hard, "Hard — predicted")
plt.tight_layout(); plt.show()

## Compute pixel-level metrics (IoU, Dice)

**What's happening here.** *Intersection over Union* (IoU) is the area where prediction and truth agree, divided by the area covered by either. *Dice* is twice the agreement divided by the sum of the two areas — equivalent to the harmonic mean of precision and recall over pixels. Both range from 0 (no overlap) to 1 (perfect overlap). Both are *pixel*-level — they ignore whether the pixels group into the right number of cells.

**Predict before running.** Imagine two binary masks that are *identical except shifted by 1 pixel in both x and y*. The cells are perfectly recovered, just slightly offset. What IoU do you expect?

- (a) ≥ 0.99 — the masks are nearly identical, so IoU should be near-perfect.
- (b) ~ 0.95 — small offset, small drop.
- (c) ~ 0.85 — most pixels overlap, but boundary pixels don't.
- (d) < 0.7 — surprisingly low for a "small" shift.

The right answer for a circular mask of radius ~12 px is closer to (c). Most students predict (a) or (b). IoU is harsher than intuition suggests because it compares *boundaries*, and boundaries are where errors live.

In [ ]:
def iou_dice_binary(pred_mask, gt_mask):
    pred_bin = pred_mask > 0
    gt_bin = gt_mask > 0
    intersection = (pred_bin & gt_bin).sum()
    union = (pred_bin | gt_bin).sum()
    pred_sum = pred_bin.sum()
    gt_sum = gt_bin.sum()
    iou = intersection / union if union > 0 else 0.0
    dice = (2 * intersection) / (pred_sum + gt_sum) if (pred_sum + gt_sum) > 0 else 0.0
    return iou, dice

iou_e, dice_e = iou_dice_binary(masks_easy, gt_easy)
iou_h, dice_h = iou_dice_binary(masks_hard, gt_hard)

print(f"Easy image  : IoU = {iou_e:.3f}, Dice = {dice_e:.3f}")
print(f"Hard image  : IoU = {iou_h:.3f}, Dice = {dice_h:.3f}")

**Where do those numbers live spatially?** The IoU/Dice scores summarize the whole image into a single number. The next cell breaks that number out across pixels: **green = true positive** (pred and GT agree there's a cell), **red = false positive** (pred says cell, GT says background), **blue = false negative** (GT says cell, pred missed it). The red and blue regions are *exactly* what's pulling the IoU down.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(pixel_agreement_rgb(masks_easy, gt_easy))
axes[0].set_title(f"Easy — IoU={iou_e:.3f}, Dice={dice_e:.3f}"); axes[0].axis('off')
axes[1].imshow(pixel_agreement_rgb(masks_hard, gt_hard))
axes[1].set_title(f"Hard — IoU={iou_h:.3f}, Dice={dice_h:.3f}"); axes[1].axis('off')
fig.suptitle("Pixel agreement map: green=TP, red=FP, blue=FN", y=1.02)
plt.tight_layout(); plt.show()

**What you should be seeing.** The easy image scores near 1.0 (the agreement map is almost entirely green). The hard image is a bit lower but still typically above 0.85 — yet the agreement map shows visible red and blue patches. Pixel metrics make problems look *less* severe than they are. The number that *seems* high (0.92!) hides the fact that *count* is off, *which cells were detected* is wrong, and the failure-mode distribution is irregular. Watch what instance-level metrics show next.

**Verify the 1-pixel-shift intuition.** Run the cell below to confirm the prediction question. A circular mask shifted by 1 pixel each way drops to IoU ≈ 0.85, not the 0.99+ most people expect.

In [ ]:
# Verify: 1-pixel shift on a circular mask
shift = 1  # @param {type: "slider", min: 0, max: 5, step: 1}

Y, X = np.ogrid[:200, :200]
mask_a = ((Y - 100)**2 + (X - 100)**2 <= 12**2).astype(np.uint8)
mask_b = ((Y - (100 + shift))**2 + (X - (100 + shift))**2 <= 12**2).astype(np.uint8)
iou_shift, dice_shift = iou_dice_binary(mask_a, mask_b)
print(f"Shift = {shift} px → IoU = {iou_shift:.3f}, Dice = {dice_shift:.3f}")

# Visualize: red = mask A only, blue = mask B only, magenta = both
overlay = np.zeros((*mask_a.shape, 3), dtype=float)
overlay[..., 0] = mask_a.astype(float)
overlay[..., 2] = mask_b.astype(float)
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(overlay)
ax.set_title(f"Mask A (red), Mask B (blue), overlap (magenta)\nshift={shift}px → IoU={iou_shift:.3f}")
ax.axis('off')
plt.tight_layout(); plt.show()

## Compute instance-level metrics

**What's happening here.** Instance metrics treat each cell as a discrete object. We match each predicted cell to a ground-truth cell if their IoU exceeds a threshold. Matched pairs are *true positives*; unmatched predictions are *false positives*; unmatched ground-truth cells are *false negatives*. **The threshold matters.** A threshold of 0.5 is the COCO/instance-segmentation convention; 0.75 is much stricter.

**Predict before running.** With `iou_threshold=0.5`, a predicted mask with IoU exactly 0.51 against the wrong ground-truth cell is counted as a true positive. What does that mean for the precision/recall numbers you'll see?

- (a) Nothing — the threshold is just a convention, the metric is robust.
- (b) The precision/recall numbers are *upper bounds*; with a stricter threshold the same predictions look worse.
- (c) The threshold is irrelevant; what matters is the underlying overlap.
- (d) The threshold should be chosen to match the biological tolerance — for sparse cells that touch, lower; for dense touching cells where boundary precision matters, higher.

(b) and (d) are both correct. The threshold is not a property of your model; it is a property of *what counts as a successful detection*, and that is a biology question, not a model question.

In [ ]:
def match_instances(pred_mask, gt_mask, iou_threshold=0.5):
    """Match each predicted instance to a GT instance by IoU >= threshold.

    Returns matched count, false positives, false negatives.
    """
    pred_ids = [i for i in np.unique(pred_mask) if i != 0]
    gt_ids   = [i for i in np.unique(gt_mask) if i != 0]
    matched_pred = set()
    matched_gt = set()

    for p in pred_ids:
        pred_bin = (pred_mask == p)
        best_iou, best_g = 0.0, None
        for g in gt_ids:
            if g in matched_gt: continue
            gt_bin = (gt_mask == g)
            inter = (pred_bin & gt_bin).sum()
            union = (pred_bin | gt_bin).sum()
            if union == 0: continue
            iou_val = inter / union
            if iou_val > best_iou:
                best_iou, best_g = iou_val, g
        if best_iou >= iou_threshold:
            matched_pred.add(p)
            matched_gt.add(best_g)

    tp = len(matched_pred)
    fp = len(pred_ids) - tp
    fn = len(gt_ids) - len(matched_gt)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall}

m_easy = match_instances(masks_easy, gt_easy)
m_hard = match_instances(masks_hard, gt_hard)
print("Easy:", m_easy)
print("Hard:", m_hard)

**Visualize the match.** Two views: (1) per-cell match status painted on each image — *green = TP, red = FP, blue = FN*; (2) a bar chart of TP/FP/FN counts side-by-side. The hard image will have visibly red and blue cells; the easy image should be almost all green.

In [ ]:
# Per-cell match status maps
status_pred_e, status_gt_e = instance_match_status(masks_easy, gt_easy, iou_threshold=0.5)
status_pred_h, status_gt_h = instance_match_status(masks_hard, gt_hard, iou_threshold=0.5)

# 4-color status colormap: 0=bg(black), 1=TP(green), 2=FP(red), 3=FN(blue)
_status_cmap = ListedColormap(['black', '#2ca02c', '#d62728', '#1f77b4'])

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(status_pred_e, cmap=_status_cmap, vmin=0, vmax=3); axes[0, 0].set_title("Easy — predicted (TP green / FP red)"); axes[0, 0].axis('off')
axes[0, 1].imshow(status_gt_e,   cmap=_status_cmap, vmin=0, vmax=3); axes[0, 1].set_title("Easy — ground truth (TP green / FN blue)"); axes[0, 1].axis('off')
axes[1, 0].imshow(status_pred_h, cmap=_status_cmap, vmin=0, vmax=3); axes[1, 0].set_title("Hard — predicted (TP green / FP red)"); axes[1, 0].axis('off')
axes[1, 1].imshow(status_gt_h,   cmap=_status_cmap, vmin=0, vmax=3); axes[1, 1].set_title("Hard — ground truth (TP green / FN blue)"); axes[1, 1].axis('off')
plt.tight_layout(); plt.show()

# Bar chart of TP/FP/FN counts
fig, ax = plt.subplots(figsize=(7, 4))
labels = ['TP', 'FP', 'FN']
x = np.arange(len(labels))
ax.bar(x - 0.2, [m_easy['tp'], m_easy['fp'], m_easy['fn']], 0.4, label='Easy', color='#4C72B0')
ax.bar(x + 0.2, [m_hard['tp'], m_hard['fp'], m_hard['fn']], 0.4, label='Hard', color='#C44E52')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Count of instances')
ax.set_title('Instance-level outcomes at IoU threshold 0.5')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

**What you should be seeing.** Pixel IoU was 0.95+ on the hard case while instance recall is now visibly lower. The same model output, scored two ways, can look excellent or mediocre depending on the metric. **The metric you choose changes how bad the problem looks.** Reporting a single IoU number in a methods section without disclosing the threshold and the matching strategy is the bioimage-analysis equivalent of selective reporting.

**Free exploration — slide the IoU threshold.** Watch how precision and recall change as you raise the threshold from 0.5 toward 0.9. Most published instance-level numbers are at threshold 0.5; the same model at 0.75 typically looks dramatically worse — and that stricter threshold is the one that matters when boundaries are biologically meaningful.

In [ ]:
# @title Instance-matching threshold exploration { run: "auto" }
iou_threshold = 0.5  # @param {type: "slider", min: 0.3, max: 0.95, step: 0.05}

m_easy_t = match_instances(masks_easy, gt_easy, iou_threshold=iou_threshold)
m_hard_t = match_instances(masks_hard, gt_hard, iou_threshold=iou_threshold)
print(f"At iou_threshold={iou_threshold}:")
print(f"  Easy: precision={m_easy_t['precision']:.2f}, recall={m_easy_t['recall']:.2f}")
print(f"  Hard: precision={m_hard_t['precision']:.2f}, recall={m_hard_t['recall']:.2f}")

**See the threshold dependence as a curve.** A single number at one threshold is not a model property — it is a *choice*. The cell below sweeps the threshold from 0.3 to 0.95 and plots precision and recall for both images. The vertical line marks the slider value above. Slide the slider, re-run, watch the line move. **Most published papers report the value at threshold 0.5; you can see how dramatically the same model looks worse at 0.75.**

In [ ]:
ts = np.linspace(0.3, 0.95, 27)
prec_e, rec_e, prec_h, rec_h = [], [], [], []
for t in ts:
    me = match_instances(masks_easy, gt_easy, iou_threshold=t)
    mh = match_instances(masks_hard, gt_hard, iou_threshold=t)
    prec_e.append(me['precision']); rec_e.append(me['recall'])
    prec_h.append(mh['precision']); rec_h.append(mh['recall'])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ts, prec_e, '-', color='#4C72B0', label='Easy precision')
ax.plot(ts, rec_e, '--', color='#4C72B0', label='Easy recall')
ax.plot(ts, prec_h, '-', color='#C44E52', label='Hard precision')
ax.plot(ts, rec_h, '--', color='#C44E52', label='Hard recall')
ax.axvline(iou_threshold, color='gray', linestyle=':', label=f'slider = {iou_threshold}')
ax.set_xlabel('IoU threshold for instance match')
ax.set_ylabel('Precision / recall')
ax.set_title('Instance metrics depend on the IoU threshold you chose')
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='lower left')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Compute biological measurements

In [ ]:
def biological_summary(pred, gt, label):
    pred_count = int(np.unique(pred[pred > 0]).size)
    gt_count = int(np.unique(gt[gt > 0]).size)
    return {
        "image": label,
        "predicted_count": pred_count,
        "true_count": gt_count,
        "count_error": pred_count - gt_count,
    }

bio_easy = biological_summary(masks_easy, gt_easy, "easy")
bio_hard = biological_summary(masks_hard, gt_hard, "hard")
df_bio = pd.DataFrame([bio_easy, bio_hard])
print(df_bio)

**Count comparison.** A bar chart that puts true and predicted counts next to each other. The gap *is* the count error; the gap is what matters biologically. Compare this to the instance precision/recall plot above — you'll see they tell different stories about the same prediction.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(df_bio))
ax.bar(x - 0.2, df_bio['true_count'],      0.4, label='True count',      color='#2c7bb6')
ax.bar(x + 0.2, df_bio['predicted_count'], 0.4, label='Predicted count', color='#d7191c')
for i, row in df_bio.iterrows():
    err = int(row['count_error'])
    ax.annotate(f"err = {err:+d}", (i, max(row['true_count'], row['predicted_count']) + 0.5),
                ha='center', fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(df_bio['image'])
ax.set_ylabel('Cell count')
ax.set_title('True vs predicted cell counts')
ax.set_ylim(0, max(df_bio[['true_count', 'predicted_count']].values.max(), 1) * 1.25)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## The metric–biology plot

**What's happening here.** This is the lab's central learning moment — the figure the morning lecture promised. We construct a series of perturbed predictions starting from ground truth and progressively deleting cells. For each perturbation we compute *both* IoU (a pixel metric) and absolute count error (a biological metric). Then we plot count error against IoU. If the two metrics tracked the same underlying quality, they would line up on a straight curve. They don't.

**Predict before running.** As perturbation strength increases (more cells deleted), what relationship do you expect between IoU and count error?

- (a) IoU drops smoothly and count error rises smoothly — the two are tightly coupled.
- (b) IoU drops *faster* than count error rises — pixel metrics are the more sensitive signal.
- (c) IoU stays high while count error rises — pixel metrics hide instance-level failures.
- (d) Both stay roughly constant; the perturbations don't perturb anything meaningful.

If you predicted (c), you've internalized the lecture. The figure should make this visceral.

In [ ]:
# Visualize the gap: high IoU does not mean a correct count.
import seaborn as sns

# Build a small set of perturbed predictions to plot a curve
rng = np.random.default_rng(0)
n_trials = 30
perturbations = np.linspace(0.0, 0.8, n_trials)

ious_curve = []
count_errors_curve = []
for p_strength in perturbations:
    test = gt_easy.copy()
    # Perturb: randomly delete a fraction of cells, randomly merge a few
    cell_ids = [i for i in np.unique(test) if i != 0]
    n_delete = int(p_strength * len(cell_ids) * 0.5)
    for cell in rng.choice(cell_ids, size=min(n_delete, len(cell_ids)), replace=False):
        test[test == cell] = 0
    iou_val, _ = iou_dice_binary(test, gt_easy)
    bio = biological_summary(test, gt_easy, "perturbed")
    ious_curve.append(iou_val)
    count_errors_curve.append(abs(bio["count_error"]))

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ious_curve, count_errors_curve, c=perturbations, cmap='viridis', s=60, edgecolor='k', linewidth=0.5)
ax.set_xlabel("IoU (pixel overlap)")
ax.set_ylabel("Absolute count error")
ax.set_title("High IoU does not guarantee a correct cell count")
ax.invert_xaxis()
cbar = plt.colorbar(ax.collections[0], ax=ax, label="Perturbation strength")
plt.tight_layout(); plt.show()

**What you should be seeing.** Count error rises faster than IoU drops. Models with IoU ≈ 0.95 can be off by *several cells*, and models with IoU ≈ 0.85 can be off by far more. The two metrics measure related but different things: IoU measures *pixel-level overlap*; count error measures *instance-level recovery*. **They can disagree by a wide margin.** Answer (c) was correct.

This is the metrics-versus-biology gap. It is the single most reported-incorrectly aspect of bioimage AI papers, and the central pedagogy of this lab.

## The key insight

A model can be 'good' by IoU and still produce systematically wrong biological measurements. **Choose validation metrics that match the biological question:**

- *How many cells?* → count error
- *What is the mean cell size?* → mean-area error
- *Is the size distribution different between conditions?* → distribution-comparison test (KS, Earth Mover's, etc.)
- *Is the model spatially biased?* → per-region precision/recall

Lab 2's central message: don't take the IoU number from a paper at face value. Ask whether it's measuring the question you actually care about.

## Closing reflection

Lab 2 makes the metrics-versus-biology gap visible. The next lab (Lab 3) extends the responsible-use thread to a different challenge: how do you validate a model output when, by definition, you don't have a clean version to compare against?

That challenge is what AI restoration (Lab 3a) and foundation-model segmentation (Lab 3b) both face. See you there.